In [ ]:
%reset -f

In [ ]:
import pypsa
import pandas as pd
import os
import matplotlib.pyplot as plt

In [ ]:
#savesth e trouble of restarting kernel, each time imported file changes
%load_ext autoreload
%autoreload 2

import src.analysis as analysis
import src.network as network
import src.plotting as plotting
import src.utils as utils

In [ ]:
data_path = os.path.join(os.getcwd(), 'data')

In [ ]:
File_name = 'proto1'
n = pypsa.Network(os.path.join(data_path, f'{File_name}.nc'))

In [ ]:
# verifying heat pump cop working well
hp_thermal = -1 * n.links_t.p1['Residential_HeatPump']

hp_env = n.links_t.p['Residential_HeatPump']
hp_thermal / hp_env


## Visualization

### Electric

In [ ]:
el_buses = n.buses[n.buses.carrier == 'AC'].index
el_gens = n.generators.loc[n.generators['bus'].isin(el_buses)]
el_gens = el_gens.loc[el_gens['carrier'] != 'Load_shed'] #removing slack gens from the list of electrical gens

el_load_shed = n.generators.loc[n.generators['carrier'] == 'Load_shed'] #:)



In [ ]:
# Determining the energy balance at each bus
bus_sizes, bus_sizes_norm = network.calc_bus_sizes(n)


#normalised bus sizes, to get a sense of the relative contribution of each carrier at each bus


In [ ]:
n.generators

In [ ]:
(n.generators.groupby('bus').apply(lambda x: x.index.tolist()).to_dict()) #groups or split by bus, but since we have no operation to be applied here, converting to a dictionary wiht a lambda function

In [ ]:
n.generators_t.p['grid_export'].sum()

In [ ]:
bus_sizes

In [ ]:
fig, ax = plt.subplots(figsize = (20,8))
plotting.plot_EnBalance(n, 'AC', ax, bus_sizes = bus_sizes, title = 'Electrical energy balance', bus_scale = 1e-9)

In [ ]:
fig, ax = plt.subplots(figsize = (20,8))
plotting.plot_EnBalance(n, 'AC', ax, bus_sizes = bus_sizes_norm, title = 'Electrical energy balance', bus_scale = 1e-5)

In [ ]:
fig, ax = plt.subplots(figsize = (20,15))
plotting.plot_EnBalance(n, 'thermal', ax, bus_sizes = bus_sizes, title = 'Thermal energy balance', bus_scale = 1e-9)

In [ ]:
fig, ax = plt.subplots(figsize = (20,15))
plotting.plot_EnBalance(n, 'thermal', ax, bus_sizes = bus_sizes_norm, title = 'Thermal energy balance', bus_scale = 1e-5)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 15), nrows = 3)
plotting.plot_EnBalance(n, 'AC', ax[0], bus_sizes = bus_sizes, title = 'Electrical energy balance', bus_scale = 1e-10)
plotting.plot_EnBalance(n, 'thermal', ax[1], bus_sizes = bus_sizes_norm, bus_scale = 1e-7, title = 'Thermal energy balance')
plotting.plot_EnBalance(n, 'gas', ax[2], bus_sizes = bus_sizes_norm, bus_scale = 1e-7, title = 'Gas energy balance')

### Component loading

In [ ]:
import numpy as np
from adjustText import adjust_text #to print text without overlaps automatically
import matplotlib.cm as cm
import matplotlib.colors as colors

In [ ]:
def get_carrier_Xlinks(n, carrier):
    """
    Function to get the X-coupling links connected with buses of given carrier.
    considers if either of the two output buses is connected to a bus of the given carrier
    """
    buses = n.buses[n.buses.carrier == carrier].index
    # Transport links have carriers like AC or gas
    Xlinks = n.links.loc[((n.links.carrier != 'AC') & (n.links.carrier != 'gas')) & ((n.links.bus1.isin(buses)) | (n.links.bus2.isin(buses)))]
    return Xlinks

**Setup instruction**
The carrier of transport links have to be set similar to that of the buses it connections.  
Leaving it empty, would make it default to this.  
Xlinks should have carreirs defined by their techs.  


In [ ]:
def get_carrieronly_network(n, carrier):
    '''
    Function that would return entities associated with buses of a given carrier.
    Currently the returns are:
    - Buses: all buses of the given carrier
    - gens: all gens connected to buses of the given carrier
    - loads: all loads connected to buses of the given carrier
    - links: all transport links of the given carrier
    - Xlinks: all sector-coupling links connected to buses of the given carrier
    - Stores: all stores connected to buses of the given carrier
    '''

    cr_buses = n.buses[n.buses.carrier == carrier].index
    trans_links = n.links[n.links.carrier == carrier].index
    gens = n.generators[(n.generators.bus.isin(cr_buses)) & (n.generators.carrier != 'Load_shed')].index
    loads = n.loads[n.loads.bus.isin(cr_buses)].index
    Xlinks = get_carrier_Xlinks(n, carrier).index
    stores = n.stores[n.stores.bus.isin(cr_buses)].index
    slacks = n.generators[(n.generators.carrier == 'Load_shed') & (n.generators.bus.isin(cr_buses))].index
    if carrier == 'thermal':
        trans_links = n.links[n.links.carrier == 'DHN'].index
    network_slice = {
        'buses': cr_buses,
        'gens': gens,
        'trans_links': trans_links,
        'loads': loads,
        'xlinks': Xlinks,
        'stores': stores,
        'slacks': slacks
    }

    return network_slice

In [ ]:
carrier = 'AC'
n.buses[n.buses.carrier == carrier]

In [ ]:
def compute_comp_loading(n, carrier='all'):
    '''
    Function to compute the loading of all components of a given carrier, and returns a dataframe.
    Return:
    A dictionary with keys of component types, and values with dataframes of loading for each snapshot.
    example:
    {
        'gens': pd.DataFrame(...),}
    '''

    if carrier == 'all': #untested, but should work
        loading = {}
        loading['gens'] = (n.generators_t.p.abs() / n.generators['p_nom'])
        xlinks = n.links[((n.links.carrier != 'AC') & (n.links.carrier != 'gas') & (n.links.carrier != 'thermal'))].index
        loading['trans_links'] = (n.links_t.p1.abs() / n.links['p_nom']).loc[:,n.links.index.difference(xlinks)]
        #sector-coupling links loading, using p0 instead of p1, p_nom limits p0, hence this is more reasonable
        loading['xlinks'] = (n.links_t.p0.abs() / n.links['p_nom']).loc[:,xlinks]
        
    else:
        cr_network = get_carrieronly_network(n, carrier) #calling this method here makes it more easier for user, if they wish to calculate loading on all components irrespective of carrier
        loading = {}
        loading['gens'] = (n.generators_t.p[cr_network['gens']].abs() / n.generators.loc[cr_network['gens'], 'p_nom'])
        loading['trans_links'] = (n.links_t.p1[cr_network['trans_links']].abs() / n.links.loc[cr_network['trans_links'], 'p_nom'])
        #sector-coupling links loading, using p0 instead of p1, p_nom limits p0, hence this is more reasonable
        loading['xlinks'] = (n.links_t.p0[cr_network['xlinks']].abs() / n.links.loc[cr_network['xlinks'], 'p_nom'])

        
        return loading

In [ ]:
def prep_colour_scheme(quantity, thresholds = {'Default' : 1}, get_cmap = False):
    '''
    Function to prepare a colour scheme according to the provided loading and specified thresholds.
    Threshold being used to consider appropriate loading for heat pumps; 
    could also be used to set 0.75 or something as the threshold for transport lines (N-1 criteria)
    '''

    cmap = plt.get_cmap('bwr')
    norm = colors.Normalize(vmin=0, vmax=thresholds.get('Default', 1))
    

    col_scheme = {
        comp: {
            entity: cmap(norm(quantity[comp][entity])) for entity in quantity[comp].keys()
        }for comp in quantity.keys()
    }
    col_scheme['bus'] = 'orange'  # Keeping bus color constant
    # now the seperate threshold for heat pump
    if get_cmap is False:
        return col_scheme
    else:
        return col_scheme, cmap, norm

In [ ]:
def plot_network(ax, n, carrier, show_slacks = False, show_HTLTlinks = False,col_scheme = None, title = None, bus_marker_size = 100, gen_marker_size = 50, link_linewidth = 1, spread = 0.002, debug = False ):
    """
    Plots the network of the specified carrier with the specified colour scheme.
    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes on which the network is drawn.
    n : pypsa.Network
        PyPSA network containing buses, generators and links.
    carrier : str
        Carrier whose network topology should be visualized (e.g. "AC", "heat").
    show_slacks : bool, default=False
        If True, include slack generators in the plot.
    col_scheme : dict, optional
        Dictionary specifying colours for generators, links, buses and sector-coupling links.
    title : str, optional
        Title of the plot.

    """
    if col_scheme is None: #better practice to set mutable defaults inside function, else changes made here, would pass to other calls
        col_scheme = {'gens' : {}, 'xlinks' : {}, 'bus' : 'orange', 'trans_links' : {}}

    debug_log = []
    unqbus = []

    # Slicing the network for the given carrier, and unpacking
    cr_network = get_carrieronly_network(n, carrier)
    gens = cr_network['gens']
    Xlinks = cr_network['xlinks']
    transport_links = cr_network['trans_links']
    buses = cr_network['buses']

    if show_HTLTlinks is False:
        Xlinks = [link for link in Xlinks if n.links.loc[link, 'carrier'] != 'HT_LT_link']

    #Aggregating generators and X-coupling links per bus, to plot them radially around the bus
    if show_slacks is False:
        gens_per_bus = n.generators.loc[gens].groupby('bus').groups
    else:
        gens_per_bus = n.generators.groupby('bus').groups
    out_buses = n.links.loc[Xlinks][['bus1', 'bus2']].stack().reset_index(level = 1, drop = True).rename('out_buses')
    out_buses = out_buses[out_buses != ''] #removing the empty buses
    Xlinks_per_bus = out_buses.groupby(out_buses).groups #Duplication is not an issue, since we are iterating through buses to plot
    
    
    #Plotting buses
    ax.scatter(n.buses.loc[buses].x, n.buses.loc[buses].y, s = bus_marker_size, color = col_scheme.get('bus'), label = 'Buses', zorder = 3)
    debug_log.append(f'{len(buses)} buses in total: \t {buses}\n\n')
    #Plotting transport links or lines
    for link in transport_links:
        link_x0 = n.links.bus0.map(n.buses.x)[link]
        link_y0 = n.links.bus0.map(n.buses.y)[link]
        link_x1 = n.links.bus1.map(n.buses.x)[link]
        link_y1 = n.links.bus1.map(n.buses.y)[link]
        ax.plot([link_x0, link_x1], [link_y0, link_y1], color = col_scheme.get('trans_links').get(link, 'grey'), alpha = 1, linewidth = link_linewidth)
    debug_log.append(f'{len(transport_links)} transport links plotted: \t {transport_links}\n\n')
    
    texts = [] #all text collexted here, and managed by adjust_text to avoid overlaps
    # offset = 80e-6
    offset = 0
    #Plotting generators and sector-coupling gens, and distributing them radially around the bus
    radius = spread
    angles_gens = {}
    angles_xlinks = {}
    ##Plotting gens and Xlilnks bus-wise
    for bus in buses:
        bus_gens = gens_per_bus.get(bus)
        bus_Xlinks = Xlinks_per_bus.get(bus)
        
        bus_x = n.buses.loc[bus].x
        bus_y = n.buses.loc[bus].y

        #Annotation for bus
        
        name = bus.strip('_HT').strip('_LT').strip('_gas')  # Stripping suffixes to get the base name
        if name not in unqbus:
            unqbus.append(name)
            texts.append(ax.text(bus_x + offset, bus_y + offset, name, fontsize = 8))
            debug_log.append(f'Bus annotation added: {name}')
            

        
        ## What happens if there are no gens or Xlinks connected to the bus? 
        if bus_gens is not None and len(bus_gens) > 0:
            debug_log.append(f'Adding gens to bus {bus}:')
            angles_gens = np.linspace(0, np.pi, len(bus_gens), endpoint=False)
            for gen, angle in zip(bus_gens, angles_gens):
                gen_x = bus_x + radius * np.cos(angle)
                gen_y = bus_y + radius * np.sin(angle)
                ax.scatter(gen_x, gen_y, s = gen_marker_size, color = col_scheme.get('gens').get(gen, 'green'), label = 'Generators')
                ax.plot([bus_x, gen_x], [bus_y, gen_y], color = 'grey', alpha = 0.5)
                texts.append(ax.text(gen_x + offset, gen_y + offset, gen, fontsize = 8))
                debug_log.append(f'\tGenerator {gen} plotted.')

        
        if bus_Xlinks is not None and len(bus_Xlinks) > 0:
            debug_log.append(f'Adding Xlinks to bus {bus}:')
            angles_xlinks = np.linspace(np.pi, 2*np.pi, len(bus_Xlinks), endpoint=False)
            for Xlink, angle in zip(bus_Xlinks, angles_xlinks):
                Xlink_x = bus_x + radius * np.cos(angle)
                Xlink_y = bus_y + radius * np.sin(angle)
                ax.scatter(Xlink_x, Xlink_y, s = gen_marker_size, color = col_scheme.get('xlinks').get(Xlink, 'teal'), label = 'X-coupling links')
                ax.plot([bus_x, Xlink_x], [bus_y, Xlink_y], color = 'grey', alpha = 0.5)
                texts.append(ax.text(Xlink_x + offset, Xlink_y + offset, Xlink, fontsize = 8))
                debug_log.append(f'\tXlink {Xlink} plotted.\n')

    debug_log.append(f'\n\n\t\t {len(unqbus)} bus annotations added\n')
    # adjust_text(texts, ax = ax, arrowprops=dict(arrowstyle='->', color='red'))
    texts = list({t.get_text():t for t in texts}.values())  # Remove duplicates based on text content, dict retains only last occurence of the key, values are the text objects, which are then passed back
    if debug:
        print("\n".join(debug_log))
    adjust_text(texts, ax = ax)

    ax.set_xticks([])
    ax.set_yticks([])
    
    #removing duplicate legends
    # handles, labels = ax.get_legend_handles_labels()
    # by_label = dict(zip(labels, handles))
    # ax.legend(by_label.values(), by_label.keys())

    if title is not None:
        ax.set_title(title)

    



In [ ]:
def plot_90th_percentile_loading(n, carrier, ax, comp_loading = None, title = None, **kwargs):
    """
    Funciton to plot the 90th percentile loading of all components of specified carrier.
    Parameters:
    ----------
    n : pypsa.Network
    carrier : str, Carrier whose network topology should be visualized (e.g. "AC", "heat").
    ax : matplotlib.axes.Axes
    comp_loading: dict, optional, default = None
    Dictionary containing dataframes of loading for each component type. If None, will be computed, 
    provides user flexibility to slice the component loading dataframes.
    
    """
    if comp_loading is None:
        comp_loading = compute_comp_loading(n, carrier)

    comp_loading_90th = {
        comp: {
            entity: comp_loading[comp][entity].quantile(0.9) for entity in comp_loading[comp].columns
        }for comp in comp_loading.keys()
    }

    col_scheme, cmap, norm = prep_colour_scheme(comp_loading_90th, get_cmap = True)

    plot_network(ax, n, carrier, col_scheme = col_scheme, title = title, **kwargs)

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])   # required by matplotlib

    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label("90th percentile loading")


In [ ]:
fig, ax = plt.subplots(figsize = (15,8))
loading = compute_comp_loading(n, 'AC')
# loading = {
#     comp: df.loc[:'06-2026', :] for comp, df in loading.items()
# }
plot_90th_percentile_loading(n, 'AC', ax, loading, title = 'Electrical network 90th percentile loading')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (15,8))

# loading = {
#     comp: df.loc[:'06-2026', :] for comp, df in loading.items()
# }
plot_90th_percentile_loading(n, 'thermal', ax, title = 'Thermal network 90th percentile loading', spread = 0.008, debug = False)
plt.show()

In [ ]:
#Plotting only 90th percentile of load shedding snaps

fig, ax = plt.subplots(figsize = (15,8))
loading = compute_comp_loading(n, 'thermal')

slack_gens = n.generators_t.p[get_carrieronly_network(n, 'thermal')['slacks']]
mask = (slack_gens > 0).any(axis = 1)
load_shedding_snaps = n.snapshots[mask]
loading = {
    comp: df.loc[load_shedding_snaps, :] for comp, df in loading.items()
}
plot_90th_percentile_loading(n, 'thermal', ax, loading, title = 'Thermal network 90th percentile loading during load-shedding')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (15,8))
plot_90th_percentile_loading(n, 'gas', ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (15,8))

plot_network(ax, n, 'thermal', show_slacks=False)

In [ ]:
n.links[n.links['carrier'] == 'HP']

In [ ]:
#computing component loading

threshold = 1
carrier = 'thermal'

cr_network = get_carrieronly_network(n, carrier)

#filtering snapshots where load shedding is happening, anywhere

slack_gens = n.generators_t.p[cr_network['slacks']]
mask = (slack_gens > 0).any(axis=1)
load_shedding_snaps = n.snapshots[mask]

loading = compute_comp_loading(n, carrier) 

#preparing colour scheme based on thresholds
## 90th percentile loading during load shedding snaps

# shedding_loading_90th = {comp: {entity: loading[comp][entity].loc[load_shedding_snaps].quantile(0.9)} for comp in loading.keys() for entity in loading[comp].keys()}
shedding_loading_90th = {
    comp: {
        entity: loading[comp][entity].loc[load_shedding_snaps].quantile(0.9)
        for entity in loading[comp].columns}
    for comp in loading.keys()}
# or if a particular index
index = '2026-01-03 06:00:00'
idx_loading = {
    comp: {
        entity: loading[comp][entity].loc[index]
        for entity in loading[comp].columns}
    for comp in loading.keys()
    }

fig, ax = plt.subplots(figsize = (15,8))
plot_network(ax, n, carrier, col_scheme = prep_colour_scheme(shedding_loading_90th))



## Self-consumptions

In [ ]:
import plotly.graph_objects as go

grid_terminal = n.links_t.p1['grid'] #positive means import here, negative export
# export_terminal = -1 * n.stores_t.p['Export_sink'] #positive means charging, or grid export
export_terminal = n.generators_t.p['grid_export'] # positive is export
grid_gen = n.generators_t.p['Grid'] #positive means generation, negative means load


fig = go.Figure()

fig.add_trace(go.Scatter(
    x = grid_terminal.index,
    y = grid_terminal.values,
    mode = 'lines',
    name = 'Grid link',
    line = dict(color = 'green', dash = 'dashdot', shape = 'hv')
))

fig.add_trace(go.Scatter(
    x = export_terminal.index,
    y = export_terminal.values,
    mode = 'lines',
    name = 'Export terminal',
    line = dict(color = 'red', dash = 'solid', shape = 'hv')
))
fig.add_trace(go.Scatter(
    x = grid_gen.index,
    y = grid_gen.values,
    mode = 'lines',
    name = 'Grid generation',
    line = dict(color = 'purple', dash = 'solid', shape = 'hv')
))

fig.update_layout(
    height = 400,
    showlegend = True,
    xaxis = dict(title = 'Time'),
    yaxis = dict(title = 'Power (MW)'),
    
)
fig.show()

In [ ]:
# one final plot to see if everythign adds up
'''
electric flow into network considered positive, and out of network like loads, and export considered negative.
The sum should be zero, if everything adds up. The question was with the CHP el. generation, is it being used to meet the demand, or only exported out.
The plot below shows that the CHP el. generation is being used to meet the demand, and exported only when there is further excess, wherein grid import falls to zero.
Moreover, the plot also shows the difference is not exactly zero, represented on the second y axis as deviation percentage. This seems to directly correlate with 
energy flowing towards the grid/substation node, likely due to the efficiency losses in the distribution links. 
'''

plot_data = pd.DataFrame(index = n.snapshots)
plot_data['tot_el_gen'] = (-1 * (n.links_t.p2['Kritis_CHP'] + n.links_t.p2['DEP_CHP']) )+ n.generators_t.p['Roof_pv'] #link sign convention based on perspective of the bus, outwards is negative, outbus will be negative thereby, but gens follow positive for generated power
plot_data['grid'] = n.generators_t.p['Grid'] #positive means generation

plot_data['tot_el_demand'] = -1 * (n.loads_t.p[['industry', 'kritis', 'residential', 'residential2', 'residential3' ]].sum(axis = 1) + n.links_t.p.loc[:,n.links.carrier == 'HP'].sum(axis = 1)) #negative means demand, positive means generation
plot_data['export'] = -1 * n.generators_t.p['grid_export'] # negative is export

plot_data['diff'] = plot_data[['tot_el_gen', 'grid', 'tot_el_demand', 'export']].sum(axis = 1) #should be zero, if everything adds up

#diff might not be zero always, would be slightly positive, possibly due to efficiency losses in distribution links
plot_data['dev'] = plot_data['diff']/plot_data[['tot_el_demand', 'export']].abs().sum(axis = 1) * 100

start = pd.Timestamp('2026-01-01 00:00:00')
end = pd.Timestamp('2026-12-31 00:00:00')
start = pd.Timestamp('2026-06-01 00:00:00')
end = pd.Timestamp('2026-06-04 00:00:00')
plot_data = plot_data.loc[start:end]

fig, ax = plt.subplots(figsize = (15,6))

ax.step(plot_data['grid'].index, plot_data['grid'], where = 'pre', label = 'Grid', color = 'blue', linestyle = '-.')
ax.fill_between(x = plot_data['grid'].index, y1 = 0,y2 =plot_data['grid'], step = 'pre', color = 'blue', alpha = 0.2)

ax.step(plot_data['tot_el_gen'].index, (plot_data['grid']+plot_data['tot_el_gen']), where = 'pre', label = 'Total electrical generation', color = 'green', linestyle = '-')
ax.fill_between(plot_data['tot_el_gen'].index, y1 = plot_data['grid'], y2 = (plot_data['grid']+plot_data['tot_el_gen']), step = 'pre', color = 'green', alpha = 0.2)


ax.step(plot_data['export'].index, plot_data['export'], where = 'pre', label = 'Export', color = 'red', linestyle = '-.')
ax.fill_between(plot_data['export'].index, y1 = 0, y2 = plot_data['export'], step = 'pre', color = 'red', alpha = 0.2)

ax.step(plot_data['tot_el_demand'].index, (plot_data['tot_el_demand']+plot_data['export']), where = 'pre', label = 'Total electrical demand', color = 'purple', linestyle = '-')
ax.fill_between(plot_data['tot_el_demand'].index, y1 = plot_data['export'], y2 = (plot_data['tot_el_demand']+plot_data['export']), step = 'pre', color = 'purple', alpha = 0.2)

ax.plot(plot_data['diff'].index, plot_data['diff'], color = 'black', linestyle = ':', label = 'Difference')

ax2 = ax.twinx()
ax2.plot(plot_data['dev'].index, plot_data['dev'], color = 'orange', linestyle = '--', label = 'Deviation (%)')
ax2.set_ylabel('Deviation (%)')
ax2.set_ylim([-10,10])

ax.set_ylabel('Power (MW)')
ax.set_xlabel('Time')
ax.set_title('Stacked electric balance check')
ax.legend()

plt.show()


There seems to be now leakage, the surplus generation is redirected withing the network, and only the remaining is exported, at those timestamps the grid import is 0, only export.
the difference is around 7%, whihc makes sense, and can be attributed to the losses in the distribution lines.
the deviation changes, as the flow towards the grid increases, either when grid import increases, or when export increases.
But eitherways, everything checks out well here, and simply a cool plot now

## Energy balance

In [ ]:
bus2 = n.links.bus2.map(n.buses.carrier).loc['grid']
if bus2 is np.nan:
    print('When bus2 missing, then nan')
elif bus2 == '':
    print('When bus2 missing, then empty string')
else:
    print('something else')

In [ ]:
def calc_sankey_flows(n, step_sz, exclude_slacks = None, trans_links = None, Heatpump_carrier = None, export_carrier = None, grid_carrier = None):
    """
    Function to calculate the flows for a Sankey diagram based on the network's links, generators, and loads. 
    It aggregates the energy flows between different carriers, including losses and environmental contributions for heat pumps.

    Parameters:
    ----------
    n : pypsa.Network
        The PyPSA network object containing the energy system data.
    step_sz : float
        The time step size in hours for aggregating the energy flows.
    exclude_slacks : binary, optional
        If True, excludes slack generators from the calculations. Default is False.
    trans_links : list of str, optional
        List of transport link carriers, which would not be represented in the Sankey diagram. Default is ['AC', 'gas', 'DHN', 'HT_LT_link'].    
    export_carrier : str, optional
        The carrier name for the export terminal. Default is 'Export'.
    grid_carrier : str, optional
        The carrier name for the grid terminal. Default is 'grid'.


    
    """
    #setting default values for optional parameters
    if exclude_slacks is None:
        exclude_slacks = False
    if trans_links is None:
        trans_links = ['AC', 'gas', 'DHN', 'HT_LT_link']
    if Heatpump_carrier is None:
        Heatpump_carrier = 'HP'
    if export_carrier is None:
        export_carrier = 'Export'
    if grid_carrier is None:
        grid_carrier = 'grid'


    sankey_df = pd.DataFrame(columns = ['source', 'target', 'value', 'entity']) #aggregated yearly values here, step wise losses, later!!


    # X-coupling links
    xlinks = n.links.loc[~n.links.carrier.isin(trans_links)]
    for xlink in xlinks.index:
        #bus0 to tech
        src1_carrier = n.links.bus0.map(n.buses.carrier).loc[xlink]
        des1_carrier = n.links.loc[xlink, 'carrier'] #using the technology as the target, instead of the bus carrier
        val1 = n.links_t.p0[xlink].sum() * step_sz #input bus +ve
        #tech to bus1
        src2_carrier = des1_carrier
        des2_carrier = n.links.bus1.map(n.buses.carrier).loc[xlink]
        val2 = -1 * n.links_t.p1[xlink].sum() * step_sz  #output bus -ve
        #tech to bus2, nan if no bus2
        src3_carrier = des1_carrier
        des3_carrier = n.links.bus2.map(n.buses.carrier).loc[xlink]
        val3 = -1 * n.links_t.p2[xlink].sum() * step_sz #output bus -ve, would be zero if no output on bus2
        #tech to losses, if any
        if n.links.loc[xlink, 'carrier'] == Heatpump_carrier: #Heatpump considered to have no losses, instead difference represented as energy from environment
            src4_carrier = 'Environment'
            des4_carrier = des1_carrier
            val4 = -1 * (val1 - val2)
        else:
            src4_carrier = des1_carrier
            des4_carrier = 'Losses'
            val4 = val1 - val2 - val3

        sankey_df = pd.concat([sankey_df, pd.DataFrame({
            'source': [src1_carrier, src2_carrier, src3_carrier, src4_carrier],
            'target': [des1_carrier, des2_carrier, des3_carrier, des4_carrier],
            'value': [val1, val2, val3, val4],
            'entity': [xlink]*4
        }
        )]) 
    
    #gens
    gens = n.generators if exclude_slacks is False else n.generators.loc[n.generators.carrier != 'Load_shed']
    for gen in gens.index:
        source_carrier = n.generators.loc[gen, 'carrier']
        target_carrier = n.generators.bus.map(n.buses.carrier).loc[gen]
        value = n.generators_t.p[gen].sum() * step_sz
        #If export generator
        #flag: carrier mention here, if changing in data, update here as well!!
        if source_carrier == export_carrier:
            source_carrier, target_carrier = target_carrier, source_carrier #wow, crazy, tuple unpacking, only works with python!
            #setting target as Grid, to show a loop
            target_carrier = grid_carrier

        if source_carrier != target_carrier: #e.g gas to gas, at gas supply station
            sankey_df = pd.concat([sankey_df, pd.DataFrame({
                'source': [source_carrier],
                'target': [target_carrier],
                'value': [value],
                'entity': [gen]
            })])

    
    #loads
    for load in n.loads.index:
        source_carrier = n.loads.bus.map(n.buses.carrier).loc[load]
        target_carrier = n.loads.loc[load, 'carrier']
        value = n.loads_t.p[load].sum() * step_sz
        
        sankey_df = pd.concat([sankey_df, pd.DataFrame({
            'source': [source_carrier],
            'target': [target_carrier],
            'value': [value],
            'entity': [load]
        }
        )])

    #aggregating values for same source-target pairs, and dropping nans in target column (bus2 missing cases)
    sankey_df = sankey_df.groupby(['source', 'target']).agg({'value': 'sum', 'entity': lambda x: ', '.join(x)}).reset_index()
    
    # DHN Losses:
    # the techs losses already accounted, energy gen at DEP, but load at network buses, so the difference here could represent the DHN losses
    #flag: carrier mention here, if changing in data, update here as well!!
    dhn_loss = pd.DataFrame({
        'source' : ['thermal'],
        'target' : ['DHN_losses'],
        'value' : sankey_df.loc[sankey_df['target'] == 'thermal', 'value'].sum() - sankey_df.loc[sankey_df['source'] == 'thermal', 'value'].sum(),
        'entity' : ['DHN_losses']
    })

    sankey_df = pd.concat([sankey_df, dhn_loss], ignore_index=True)

    return sankey_df
    


In [ ]:
def map_label_colours(n, sankey_df, default_col = 'grey'):
    """
 
    """
    #label name from sankey_flows dataframe
    labels = pd.DataFrame(pd.concat([sankey_df['source'], sankey_df['target']]).unique(), columns = ['label'])
    #label colours from registered carriers in the network
    labels['colours'] = labels['label'].map(n.carriers['color']).fillna(default_col) #setting default colour to grey for carriers not in the network
    #setting some custom colours for losses and environment
    labels.loc[labels['label'] == 'Losses', 'colours'] = 'black' #setting losses to black
    labels.loc[labels['label'] == 'Environment', 'colours'] = '#07f8e1' #setting environment to black

    #creating a seperate column for position, since index will be used for mapping in the sankey diagram
    labels['posn'] = labels.index
    labels.set_index('label', inplace = True)
    
    return labels

In [ ]:
sankey_df = calc_sankey_flows(n, step_sz = 1, exclude_slacks = False, Heatpump_carrier = 'HP', export_carrier = 'Export', grid_carrier = 'grid')
labels = map_label_colours(n, sankey_df)
labels

In [ ]:
fig = go.Figure(data = [go.Sankey(
    arrangement = "snap",
    valueformat = ".0s",
    valuesuffix = "Wh",

    node = dict(
        pad = 20,
        thickness = 20,
        # line = dict(color = "black", width = 0.5),
        label = labels.index.to_list(),
        color = labels['colours'].to_list()
    ),
    link = dict(
        arrowlen = 5,
        source = sankey_df['source'].map(labels['posn']).to_list(),
        target = sankey_df['target'].map(labels['posn']).to_list(),
        value = (sankey_df['value'] * 1e6).to_list(), #converting to Wh, plotly automatically scales to MWh, GWh etc
        color = sankey_df['source'].map(labels['colours']).to_list()
    )

)])

fig.show()

So, this is how the carriers are being utilised (useful for documentation for vero):  
The sankey is derived like this:  
- conventional generators:  
    - gen.carrier -> bus.carrier
- sector coupling links
    - bus0.carrier -> link.carrier(technology)
    - link.carrier -> bus1.carrier
    - link.carrier -> bus2.carrier
- loads
    - bus.carrier -> load.carrier 

To increase resolution of the sankey energy balance diagram, simply use more specific carriers.  
For example, if you wish to be more specific, you could differentiate the carriers into high temp and low temp as well.  

Regular transport links are not represented in the energy balance.  

# EOF
rough

In [ ]:
n.statistics.energy_balance(bus_carrier = 'AC', groupby = ['bus', 'carrier'])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

elbusdf = n.buses.loc[el_buses]
ax.scatter(elbusdf.x, elbusdf.y)

for bus in elbusdf.index:
    ax.text(
        elbusdf.loc[bus, "x"],
        elbusdf.loc[bus, "y"],
        bus,
        fontsize=8
    )

ax.set_aspect("equal")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

for link in n.links.itertuples():
    x0, y0 = n.buses.loc[link.bus0, ["x", "y"]]
    x1, y1 = n.buses.loc[link.bus1, ["x", "y"]]

    ax.plot(
        [x0, x1],
        [y0, y1],
        color="grey",
        linewidth=1
    )

for bus in n.buses.index:
    ax.text(
        n.buses.loc[bus, "x"],
        n.buses.loc[bus, "y"],
        bus,
        fontsize=8,
        # xytext=(5,5),
        # textcoords="offset points"
    )

for bus in n.buses.index:
    ax.text(
        n.buses.loc[bus, "x"],
        n.buses.loc[bus, "y"],
        bus,
        fontsize=8,
        # xytext=(5,5),
        # textcoords="offset points"
    )

In [ ]:
fig, ax = plt.subplots(
    figsize=(12, 8),
    subplot_kw={"projection": ccrs.Mercator()}
)
n.statistics.energy_balance.plot.map( ax = ax, bus_split_circle = False)

In [ ]:
n.statistics.energy_balance(
    groupby = ["bus", "carrier"],
    components = ['Generator', 'Load', 'Link']
).groupby(['bus', 'carrier']).sum()
# n.statistics.energy_balance()

In [ ]:

import matplotlib.pyplot as plt


fig, ax = plt.subplots(figsize=(8, 8))

n.plot(
    ax=ax,
    geomap=False,          # <-- key fix: don't require cartopy GeoAxes
    bus_sizes=0.0000000000002,
    bus_colors="steelblue",
    line_widths=0,
    link_widths=2,
    link_colors="firebrick",
    title="Network topology",
)

# add bus name annotations
for name, row in n.buses.iterrows():
    ax.annotate(
        name,
        xy=(row.x, row.y),
        xytext=(5, 5),                # offset in points, so it doesn't sit on top of the marker
        textcoords="offset points",
        fontsize=8,
        ha="left",
    )

plt.tight_layout()
plt.show()

In [ ]:
n.stats.energy_balance().plot()